# 06 — Assemble a complete decoder

Our baseline is token + learned position embeddings → repeated pre-norm attention/MLP blocks → final LayerNorm → vocabulary logits.

States have shape [B,T,D]; logits have shape [B,T,V]. The attention output projection returns model-width features; the vocabulary head scores candidate next tokens. This notebook inspects the complete model before training.

## How to work through this notebook

Run setup once. At each checkpoint, write a prediction and try the small implementation before reading its adjacent reference solution. All reference cells run unchanged from top to bottom; exercise cells contain safe, optional starting points. Numerical checks use CPU float64 unless explicitly noted. Agent-verified reference execution is separate from your learning progress.

In [ ]:
from pathlib import Path
import sys, copy, math, inspect
from dataclasses import replace
import torch
from torch import nn
from torch.nn import functional as F
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/dongxi_llms/decoder_lab.py").exists()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the Dongxi_LLMs repository")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from dongxi_llms.decoder_lab import (
    DecoderConfig, TinyDecoder, DecoderBlock, MultiHeadAttention, MLP, RMSNorm,
    layer_norm, rms_norm, rope, attend, parameter_count, analytical_parameters,
    cost_estimate, teaching_batch, next_token_loss, fit_one_batch)
torch.set_num_threads(1)
torch.manual_seed(505)
DTYPE = torch.float64
def close(actual, expected, atol=1e-10, rtol=1e-8):
    torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)
print("CPU reference environment:", torch.__version__)


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
from dongxi_llms import decoder_visuals as viz
def show_visual(figure):
    display(figure)
    plt.close(figure)


## Architecture map — your location in the model

The highlighted stage is this lesson’s focus. B = batch, T = positions, D = model width, V = vocabulary size. This is a structural map, not measured activations or runtime. The baseline route adds learned position embeddings before the blocks.

![Architecture map — your location in the model. The highlighted stage is this lesson’s focus. B = batch, T = positions, D = model width, V = vocabulary size. This is a structural map, not measured activations or runtime. The baseline route adds learned position embeddings before the blocks.](../figures/chapter-05/day-05-06_assemble_decoder-architecture-map.png)

*Saved architecture schematic. The following cell regenerates it; it does not execute or train a model.*

In [ ]:
from dongxi_llms import decoder_architecture as architecture
show_visual(architecture.model_map(focus='assembly', modern=False))

## Open one repeated decoder block

Each block has two distinct sublayers, each with its own normalization and skip addition. Final normalization and the vocabulary head live outside the block stack.

![Open one repeated decoder block. Each block has two distinct sublayers, each with its own normalization and skip addition. Final normalization and the vocabulary head live outside the block stack.](../figures/chapter-05/day-05-06_assemble_decoder-architecture-detail.png)

*Saved architecture schematic. The following cell regenerates it; it does not execute or train a model.*

In [ ]:
show_visual(architecture.block_detail(focus='assembly'))

## 1. Trace every layer

Instantiate the decoder and reconstruct its forward pass explicitly. Record shapes, initialization statistics, and finiteness.

**Your prediction:** _Write it here before running the reference._

In [ ]:
cfg = DecoderConfig()
model = TinyDecoder(cfg).double()
ids, labels = teaching_batch()
# Your implementation: state = token lookup + position lookup; then loop over blocks.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
state = model.token(ids) + model.position(torch.arange(ids.shape[1]))[None]
trace = [("input states", tuple(state.shape), float(state.detach().std()))]
for i, block in enumerate(model.blocks):
    state, _, weights = block(state)
    trace.append((f"block {i}", tuple(state.shape), float(state.detach().std())))
normalized = model.final_norm(state)
logits = model.lm_head(normalized)
close(logits, model(ids))
assert logits.shape == (2, 6, 16) and torch.isfinite(logits).all()
print(trace)
print("Vocabulary logits:", logits.shape)
print("Embedding initialization std:", float(model.token.weight.detach().std()))
print(inspect.getsource(DecoderBlock.forward))

### Why this works

All blocks preserve model width; only the vocabulary head changes the final axis to vocabulary size. The .02 initialization standard deviation is this lab's explicit choice, not a universal optimum.

### Visual explanation — Locate each component in the whole model

This is a structural schematic, not a timing diagram. Attention and MLP are the two sublayers inside each repeated block; the head at the end maps model width to vocabulary size.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![Locate each component in the whole model. This is a structural schematic, not a timing diagram. Attention and MLP are the two sublayers inside each repeated block; the head at the end maps model width to vocabulary size.](../figures/chapter-05/day-05-06_assemble_decoder-visual-decoder-route.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.decoder_route(cfg))

### Visual explanation — See a vocabulary-wide score vector at every position

Each row predicts the next token from its allowed prefix. Color shows raw logits, which can be negative and are not probabilities.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![See a vocabulary-wide score vector at every position. Each row predicts the next token from its allowed prefix. Color shows raw logits, which can be negative and are not probabilities.](../figures/chapter-05/day-05-06_assemble_decoder-visual-vocabulary-logits.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.matrices([logits[0]], ['Next-token scores'], 'One input position → one score per candidate token', xlabel='Candidate token ID', ylabel='Input position'))

## 2. Distinguish tying from copying

Compare tied and untied parameter counts. If two matrices initially have equal values, are they necessarily one shared parameter?

**Your prediction:** _Write it here before running the reference._

In [ ]:
untied = TinyDecoder(replace(cfg, tied=False)).double()
# Your implementation: count parameters; inspect Parameter identity.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
assert model.token.weight is model.lm_head.weight
assert untied.token.weight is not untied.lm_head.weight
close(torch.tensor(parameter_count(untied)-parameter_count(model)), torch.tensor(cfg.vocab*cfg.width))
for candidate in (model, untied):
    assert parameter_count(candidate) == analytical_parameters(candidate.cfg)
print("Tied / untied stored parameters:", parameter_count(model), parameter_count(untied))
loss = next_token_loss(model(ids), labels)
loss.backward()
for name in ["token.weight", "blocks.0.attn.q.weight", "blocks.0.attn.k.weight",
             "blocks.0.attn.v.weight", "blocks.0.mlp.up.weight"]:
    p = dict(model.named_parameters())[name]
    assert p.grad is not None and torch.isfinite(p.grad).all()
    print(name, tuple(p.shape), float(p.grad.norm()))

### Why this works

Tying aliases the same Parameter object, so input and output paths accumulate into one gradient. Merely copying values leaves independent trainable parameters. The printed gradients verify connectivity, not learned competence.

## 3. Test causality at the model boundary

Change the last two input IDs. Which logits must remain identical? Also replay the same sequence using a growing cache.

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Your implementation: compare the unaffected prefix and concatenate cached outputs.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
with torch.no_grad():
    full = model(ids)
    changed = ids.clone(); changed[:, 4:] = 0
    close(model(changed)[:, :4], full[:, :4])
    cache, pieces = None, []
    for t in range(ids.shape[1]):
        part, cache = model(ids[:, t:t+1], caches=cache, return_cache=True)
        pieces.append(part)
    close(torch.cat(pieces, dim=1), full)
print("Full-model causality and baseline cache replay passed.")

### Why this works

Causality must survive embeddings, normalization, attention, MLPs, and the output head. Learned absolute positions must use the continuing offset during decoding, not restart at zero.

## 4. Break label shifting without mistaking it for an architectural improvement

Compute the correct next-token loss and an incorrect current-token loss. Construct an oracle that copies the input token; what would each objective reward?

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Predict the copy-oracle result before running.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
copy_logits = F.one_hot(ids, num_classes=cfg.vocab).to(DTYPE) * 20
correct = next_token_loss(copy_logits, labels)
broken = next_token_loss(copy_logits, ids)
assert broken < 1e-6 and correct > 10
print("Model's correct loss:", float(next_token_loss(model(ids), labels).detach()))
print("Copy oracle: correct loss / unshifted loss:", float(correct), float(broken))

### Why this works

A near-zero loss under unshifted labels can reward copying the visible token. This synthetic oracle is a failure demonstration, not a trained model. The correct model objective compares position t logits with token t+1.

## Takeaway and evidence boundary

Next: hold the experiment contract fixed and test whether the connected architecture can fit one consistent batch.

Companion map: [Chapter 5 pathway](../day-05/README.md). Reusable source: [decoder_lab.py](../../src/dongxi_llms/decoder_lab.py). Record your explanation and remaining questions here; the notebook's existence does not mark the lesson complete.